In [ ]:
import os
import re
import gc
import random

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score, f1_score

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

from arabert.preprocess import ArabertPreprocessor


SEED = 42

PERTURBATION_SEEDS = [
    42,
    43,
    44,
    45,
    46
]

SEVERITIES = [
    0.10,
    0.20,
    0.30
]

DIMENSIONS = [
    "Textual Accuracy",
    "Completeness",
    "Consistency",
    "Validity",
    "Understandability"
]


TRAIN_PATH = "train.Ar.csv"
TEST_PATH = "task_A_Ar_test.csv"

TEXT_COLUMN = "text"
LABEL_COLUMN = "sarcastic"


MAX_LENGTH = 128

LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32

NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01


OUTPUT_DIR = "./sarcasm_five_quality_dimensions"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


MODELS = {

    "AraBERTv2": {

        "model_name":
            "aubmindlab/bert-base-arabertv2",

        "preprocessing":
            "arabert"
    },

    "Twitter-RoBERTa-Irony": {

        "model_name":
            "cardiffnlp/twitter-roberta-base-irony",

        "preprocessing":
            "twitter_roberta"
    }
}


def set_all_seeds(seed):

    set_seed(seed)

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)


print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


train_original = pd.read_csv(
    TRAIN_PATH
)

test_original = pd.read_csv(
    TEST_PATH
)


print(
    "\nOriginal train shape:",
    train_original.shape
)

print(
    "Original test shape:",
    test_original.shape
)

print(
    "\nTrain columns:",
    train_original.columns.tolist()
)

print(
    "Test columns:",
    test_original.columns.tolist()
)


for dataset_name, dataframe in [

    ("train", train_original),
    ("test", test_original)

]:

    required = {
        TEXT_COLUMN,
        LABEL_COLUMN
    }

    missing = (
        required
        -
        set(dataframe.columns)
    )

    if missing:

        raise ValueError(
            f"{dataset_name} missing columns: {missing}"
        )


train_df = (

    train_original[
        [
            TEXT_COLUMN,
            LABEL_COLUMN
        ]
    ]

    .copy()

    .rename(
        columns={
            TEXT_COLUMN: "text",
            LABEL_COLUMN: "labels"
        }
    )
)


test_df = (

    test_original[
        [
            TEXT_COLUMN,
            LABEL_COLUMN
        ]
    ]

    .copy()

    .rename(
        columns={
            TEXT_COLUMN: "text",
            LABEL_COLUMN: "labels"
        }
    )
)


def clean_dataframe(dataframe):

    dataframe = (

        dataframe

        .dropna(
            subset=[
                "text",
                "labels"
            ]
        )

        .reset_index(drop=True)
    )

    dataframe["text"] = (

        dataframe["text"]

        .astype(str)

        .str.strip()
    )

    dataframe = (

        dataframe[
            dataframe["text"] != ""
        ]

        .reset_index(drop=True)
    )

    dataframe["labels"] = pd.to_numeric(
        dataframe["labels"],
        errors="raise"
    ).astype(int)

    return dataframe


train_df = clean_dataframe(
    train_df
)

test_df = clean_dataframe(
    test_df
)


train_label_set = set(
    train_df["labels"].unique()
)

test_label_set = set(
    test_df["labels"].unique()
)


if not train_label_set.issubset(
    {0, 1}
):

    raise ValueError(
        f"Unexpected training labels: {train_label_set}"
    )


if not test_label_set.issubset(
    {0, 1}
):

    raise ValueError(
        f"Unexpected test labels: {test_label_set}"
    )


print(
    "\nTrain shape:",
    train_df.shape
)

print(
    "Test shape:",
    test_df.shape
)


print(
    "\nTRAIN LABEL DISTRIBUTION"
)

print(
    train_df["labels"]
    .value_counts()
    .sort_index()
)


print(
    "\nTEST LABEL DISTRIBUTION"
)

print(
    test_df["labels"]
    .value_counts()
    .sort_index()
)


num_labels = 2

id2label = {
    0: "non-sarcastic",
    1: "sarcastic"
}

label2id = {
    "non-sarcastic": 0,
    "sarcastic": 1
}


train_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "official_train.csv"
    ),

    index=False
)


test_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "official_test.csv"
    ),

    index=False
)


def stochastic_count(
    target,
    rng
):

    base = int(
        np.floor(target)
    )

    fraction = (
        target
        -
        base
    )

    if rng.random() < fraction:

        base += 1

    return base


ARABIC_PATTERN = re.compile(

    r"[\u0621-\u063A"
    r"\u0641-\u064A"
    r"\u0671-\u06D3"
    r"\u06FA-\u06FC]+"
)


def get_arabic_span(
    token,
    min_len=3
):

    if not isinstance(
        token,
        str
    ):

        return None


    matches = list(

        ARABIC_PATTERN.finditer(
            token
        )
    )


    matches = [

        m

        for m in matches

        if len(
            m.group()
        ) >= min_len
    ]


    if not matches:

        return None


    return max(

        matches,

        key=lambda m:
        len(m.group())
    )


ARABIC_CHARS = list(

    "ابتثجحخدذرزسشصضطظعغفقكلمنهوي"
    "أإآؤئءىة"
)


ORTHOGRAPHIC_ALTERNATIVES = {

    "ا": {
        "أ",
        "إ",
        "آ"
    },

    "أ": {
        "ا",
        "إ",
        "آ"
    },

    "إ": {
        "ا",
        "أ",
        "آ"
    },

    "آ": {
        "ا",
        "أ",
        "إ"
    },

    "ي": {
        "ى"
    },

    "ى": {
        "ي"
    }
}


def delete_char(
    word,
    rng
):

    if len(word) < 2:

        return word

    idx = rng.randrange(
        len(word)
    )

    return (
        word[:idx]
        +
        word[idx + 1:]
    )


def insert_char(
    word,
    rng
):

    idx = rng.randrange(
        len(word) + 1
    )

    char = rng.choice(
        ARABIC_CHARS
    )

    return (
        word[:idx]
        +
        char
        +
        word[idx:]
    )


def substitute_char(
    word,
    rng
):

    idx = rng.randrange(
        len(word)
    )

    original = word[idx]

    forbidden = {
        original
    }

    forbidden.update(

        ORTHOGRAPHIC_ALTERNATIVES.get(
            original,
            set()
        )
    )


    candidates = [

        char

        for char in ARABIC_CHARS

        if char not in forbidden
    ]


    if not candidates:

        return word


    replacement = rng.choice(
        candidates
    )


    return (
        word[:idx]
        +
        replacement
        +
        word[idx + 1:]
    )


def transpose_chars(
    word,
    rng
):

    valid_positions = [

        i

        for i in range(
            len(word) - 1
        )

        if word[i] != word[i + 1]
    ]


    if not valid_positions:

        return word


    idx = rng.choice(
        valid_positions
    )


    chars = list(word)

    chars[idx], chars[idx + 1] = (

        chars[idx + 1],
        chars[idx]
    )


    return "".join(chars)


TYPO_OPERATIONS = [

    delete_char,
    insert_char,
    substitute_char,
    transpose_chars
]


def corrupt_word(
    word,
    rng
):

    for _ in range(20):

        operation = rng.choice(
            TYPO_OPERATIONS
        )


        corrupted = operation(
            word,
            rng
        )


        if corrupted != word:

            return corrupted


    return insert_char(
        word,
        rng
    )


def perturb_accuracy(
    text,
    severity,
    rng
):

    tokens = text.split()


    eligible_indices = [

        idx

        for idx, token
        in enumerate(tokens)

        if get_arabic_span(
            token
        )
        is not None
    ]


    n = len(
        eligible_indices
    )


    if n == 0:

        return text, 0, 0


    k = stochastic_count(

        severity
        *
        n,

        rng
    )


    k = min(
        k,
        n
    )


    if k == 0:

        return text, n, 0


    selected_indices = rng.sample(
        eligible_indices,
        k
    )


    changed = 0


    for idx in selected_indices:

        token = tokens[idx]

        match = get_arabic_span(
            token
        )

        original_word = (
            match.group()
        )

        corrupted_word = (
            corrupt_word(
                original_word,
                rng
            )
        )


        tokens[idx] = (

            token[:match.start()]

            +

            corrupted_word

            +

            token[match.end():]
        )


        if (
            corrupted_word
            !=
            original_word
        ):

            changed += 1


    return (
        " ".join(tokens),
        n,
        changed
    )


def perturb_completeness(
    text,
    severity,
    rng
):

    words = text.split()

    n = len(words)


    if n <= 1:

        return text, n, 0


    k = stochastic_count(

        severity
        *
        n,

        rng
    )


    k = min(
        k,
        n - 1
    )


    if k == 0:

        return text, n, 0


    selected = set(

        rng.sample(
            range(n),
            k
        )
    )


    remaining = [

        word

        for idx, word
        in enumerate(words)

        if idx not in selected
    ]


    return (
        " ".join(remaining),
        n,
        k
    )


CONSISTENCY_MAP = {

    "ا": [
        "أ",
        "إ",
        "آ"
    ],

    "أ": [
        "ا",
        "إ",
        "آ"
    ],

    "إ": [
        "ا",
        "أ",
        "آ"
    ],

    "آ": [
        "ا",
        "أ",
        "إ"
    ],

    "ي": [
        "ى"
    ],

    "ى": [
        "ي"
    ]
}


def perturb_consistency(
    text,
    severity,
    rng
):

    chars = list(text)


    eligible_indices = [

        idx

        for idx, char
        in enumerate(chars)

        if char in CONSISTENCY_MAP
    ]


    n = len(
        eligible_indices
    )


    if n == 0:

        return text, 0, 0


    k = stochastic_count(

        severity
        *
        n,

        rng
    )


    k = min(
        k,
        n
    )


    if k == 0:

        return text, n, 0


    selected_indices = rng.sample(
        eligible_indices,
        k
    )


    for idx in selected_indices:

        chars[idx] = rng.choice(

            CONSISTENCY_MAP[
                chars[idx]
            ]
        )


    return (
        "".join(chars),
        n,
        k
    )


INVALID_TOKENS = [

    "zxqv999",

    "qvzx777",

    "xqvz555",

    "vzqx333"
]


def perturb_validity(
    text,
    severity,
    rng
):

    words = text.split()

    n = len(words)


    if n == 0:

        return text, 0, 0


    k = stochastic_count(

        severity
        *
        n,

        rng
    )


    if k == 0:

        return text, n, 0


    result = words.copy()


    for _ in range(k):

        invalid_token = rng.choice(
            INVALID_TOKENS
        )

        position = rng.randrange(
            len(result) + 1
        )

        result.insert(
            position,
            invalid_token
        )


    return (
        " ".join(result),
        n,
        k
    )


def understandability_count(
    n,
    severity,
    rng
):

    if n < 2:

        return 0


    target = (
        severity
        *
        n
    )


    target = min(
        target,
        n
    )


    if target < 2:

        probability = (
            target
            /
            2
        )

        if rng.random() < probability:

            return 2

        return 0


    return min(

        stochastic_count(
            target,
            rng
        ),

        n
    )


def perturb_understandability(
    text,
    severity,
    rng
):

    words = text.split()

    n = len(words)


    if n < 2:

        return text, n, 0


    k = understandability_count(
        n,
        severity,
        rng
    )


    if k < 2:

        return text, n, 0


    selected = None


    for _ in range(20):

        candidate = sorted(

            rng.sample(
                range(n),
                k
            )
        )


        candidate_words = [

            words[i]

            for i in candidate
        ]


        if len(
            set(candidate_words)
        ) > 1:

            selected = candidate

            break


    if selected is None:

        return text, n, 0


    original_selected = [

        words[i]

        for i in selected
    ]


    permuted = None


    for _ in range(30):

        candidate = (
            original_selected.copy()
        )

        rng.shuffle(
            candidate
        )


        if (
            candidate
            !=
            original_selected
        ):

            permuted = candidate

            break


    if permuted is None:

        return text, n, 0


    output = (
        words.copy()
    )


    for idx, new_word in zip(

        selected,
        permuted

    ):

        output[idx] = (
            new_word
        )


    return (
        " ".join(output),
        n,
        k
    )


PERTURBATION_FUNCTIONS = {

    "Textual Accuracy":
        perturb_accuracy,

    "Completeness":
        perturb_completeness,

    "Consistency":
        perturb_consistency,

    "Validity":
        perturb_validity,

    "Understandability":
        perturb_understandability
}


def create_perturbed_dataframe(

    clean_df,
    dimension,
    severity,
    seed

):

    rng = random.Random(
        seed
    )


    perturb_function = (

        PERTURBATION_FUNCTIONS[
            dimension
        ]
    )


    perturbed_df = (
        clean_df.copy()
    )


    new_texts = []

    total_eligible_units = 0
    total_changed_units = 0

    changed_instances = 0


    for text in clean_df["text"]:

        (
            new_text,
            eligible_units,
            changed_units

        ) = perturb_function(

            text,
            severity,
            rng
        )


        new_texts.append(
            new_text
        )


        total_eligible_units += (
            eligible_units
        )

        total_changed_units += (
            changed_units
        )


        if new_text != text:

            changed_instances += 1


    perturbed_df[
        "text"
    ] = new_texts


    assert (

        perturbed_df[
            "labels"
        ]

        .equals(
            clean_df[
                "labels"
            ]
        )

    ), "ERROR: Gold labels changed."


    realized_rate = (

        total_changed_units
        /
        total_eligible_units

        if total_eligible_units > 0

        else 0.0
    )


    changed_instance_rate = (

        changed_instances
        /
        len(clean_df)

        if len(clean_df) > 0

        else 0.0
    )


    diagnostics = {

        "Dimension":
            dimension,

        "Severity":
            severity,

        "Seed":
            seed,

        "Eligible_Units":
            total_eligible_units,

        "Changed_Units":
            total_changed_units,

        "Realized_Rate":
            realized_rate,

        "Changed_Instances":
            changed_instances,

        "Changed_Instance_Rate":
            changed_instance_rate
    }


    return (
        perturbed_df,
        diagnostics
    )


print(
    "\n"
    +
    "=" * 90
)

print(
    "GENERATING ALL FIVE PERTURBATION CONDITIONS"
)

print(
    "=" * 90
)


PERTURBED_TEST_SETS = {}

ALL_PERTURBATION_DIAGNOSTICS = []


for dimension in DIMENSIONS:

    for severity in SEVERITIES:

        for perturb_seed in PERTURBATION_SEEDS:

            (
                perturbed_df,
                diagnostics

            ) = create_perturbed_dataframe(

                clean_df=test_df,

                dimension=dimension,

                severity=severity,

                seed=perturb_seed
            )


            key = (

                dimension,
                severity,
                perturb_seed
            )


            PERTURBED_TEST_SETS[
                key
            ] = perturbed_df


            ALL_PERTURBATION_DIAGNOSTICS.append(
                diagnostics
            )


diagnostics_df = pd.DataFrame(
    ALL_PERTURBATION_DIAGNOSTICS
)


diagnostic_summary = (

    diagnostics_df

    .groupby(
        [
            "Dimension",
            "Severity"
        ],

        as_index=False
    )

    .agg(

        Realized_Rate_Mean=(
            "Realized_Rate",
            "mean"
        ),

        Realized_Rate_SD=(
            "Realized_Rate",
            "std"
        ),

        Changed_Instance_Rate_Mean=(
            "Changed_Instance_Rate",
            "mean"
        )
    )
)


diagnostic_summary[
    "Nominal_Severity"
] = (

    diagnostic_summary[
        "Severity"
    ]
    *
    100
)


diagnostic_summary[
    "Realized_Rate_Mean"
] *= 100


diagnostic_summary[
    "Realized_Rate_SD"
] *= 100


diagnostic_summary[
    "Changed_Instance_Rate_Mean"
] *= 100


print(
    "\n"
    +
    "=" * 90
)

print(
    "PERTURBATION CHECK BEFORE TRAINING"
)

print(
    "=" * 90
)


display(

    diagnostic_summary[
        [
            "Dimension",
            "Nominal_Severity",
            "Realized_Rate_Mean",
            "Realized_Rate_SD",
            "Changed_Instance_Rate_Mean"
        ]
    ]

    .round(3)
)


diagnostic_summary.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "perturbation_check.csv"
    ),

    index=False
)


def evaluate_dataset(
    trainer,
    dataset
):

    output = trainer.predict(
        dataset
    )


    predictions = np.argmax(
        output.predictions,
        axis=-1
    )


    labels = (
        output.label_ids
    )


    accuracy = accuracy_score(
        labels,
        predictions
    )


    macro_f1 = f1_score(

        labels,
        predictions,

        average="macro",

        zero_division=0
    )


    return {

        "accuracy":
            accuracy,

        "macro_f1":
            macro_f1,

        "predictions":
            predictions,

        "labels":
            labels
    }


ALL_RESULTS = []

CLEAN_RESULTS = []


for MODEL_LABEL, CONFIG in MODELS.items():

    print(
        "\n\n"
        +
        "#" * 100
    )

    print(
        "STARTING MODEL:",
        MODEL_LABEL
    )

    print(
        "#" * 100
    )


    set_all_seeds(
        SEED
    )


    MODEL_NAME = (
        CONFIG[
            "model_name"
        ]
    )


    PREPROCESSING_TYPE = (
        CONFIG[
            "preprocessing"
        ]
    )


    if PREPROCESSING_TYPE == "arabert":

        print(
            "\nUsing AraBERT preprocessing."
        )


        arabert_preprocessor = (
            ArabertPreprocessor(

                model_name=MODEL_NAME,

                keep_emojis=True
            )
        )


        def model_preprocess(
            text
        ):

            return (

                arabert_preprocessor

                .preprocess(
                    str(text)
                )
            )


    elif PREPROCESSING_TYPE == "twitter_roberta":

        print(
            "\nUsing Twitter-RoBERTa preprocessing."
        )


        def model_preprocess(
            text
        ):

            text = str(text)

            processed_tokens = []


            for token in text.split():

                if (
                    token.startswith("@")
                    and
                    len(token) > 1
                ):

                    token = "@user"


                elif token.startswith(
                    "http"
                ):

                    token = "http"


                processed_tokens.append(
                    token
                )


            return " ".join(
                processed_tokens
            )


    else:

        def model_preprocess(
            text
        ):

            return str(text)


    print(
        "\nPreparing clean train/test inputs..."
    )


    train_model_df = (
        train_df.copy()
    )


    clean_test_model_df = (
        test_df.copy()
    )


    train_model_df[
        "text"
    ] = (

        train_model_df[
            "text"
        ]

        .apply(
            model_preprocess
        )
    )


    clean_test_model_df[
        "text"
    ] = (

        clean_test_model_df[
            "text"
        ]

        .apply(
            model_preprocess
        )
    )


    tokenizer = (

        AutoTokenizer

        .from_pretrained(
            MODEL_NAME
        )
    )


    model = (

        AutoModelForSequenceClassification

        .from_pretrained(

            MODEL_NAME,

            num_labels=2,

            id2label=id2label,

            label2id=label2id
        )
    )


    def tokenize_function(
        batch
    ):

        return tokenizer(

            batch["text"],

            truncation=True,

            max_length=MAX_LENGTH
        )


    train_dataset = (

        Dataset

        .from_pandas(

            train_model_df,

            preserve_index=False
        )

        .map(

            tokenize_function,

            batched=True
        )
    )


    clean_test_dataset = (

        Dataset

        .from_pandas(

            clean_test_model_df,

            preserve_index=False
        )

        .map(

            tokenize_function,

            batched=True
        )
    )


    data_collator = (

        DataCollatorWithPadding(
            tokenizer=tokenizer
        )
    )


    model_output_dir = (

        os.path.join(

            OUTPUT_DIR,

            MODEL_LABEL.replace(
                " ",
                "_"
            )
        )
    )


    training_args = TrainingArguments(

        output_dir=model_output_dir,

        learning_rate=(
            LEARNING_RATE
        ),

        per_device_train_batch_size=(
            TRAIN_BATCH_SIZE
        ),

        per_device_eval_batch_size=(
            EVAL_BATCH_SIZE
        ),

        num_train_epochs=(
            NUM_EPOCHS
        ),

        weight_decay=(
            WEIGHT_DECAY
        ),

        logging_strategy="epoch",

        save_strategy="no",

        report_to="none",

        seed=SEED,

        data_seed=SEED,

        optim="adamw_torch"
    )


    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=train_dataset,

        data_collator=data_collator
    )


    print(
        "\n========================================"
    )

    print(
        "TRAINING",
        MODEL_LABEL
    )

    print(
        "========================================"
    )


    trainer.train()


    clean_result = evaluate_dataset(

        trainer,

        clean_test_dataset
    )


    clean_accuracy = (
        clean_result[
            "accuracy"
        ]
    )


    clean_macro_f1 = (
        clean_result[
            "macro_f1"
        ]
    )


    CLEAN_RESULTS.append({

        "Model":
            MODEL_LABEL,

        "Clean_Accuracy":
            clean_accuracy,

        "Clean_Macro_F1":
            clean_macro_f1
    })


    print(
        "\n========================================"
    )

    print(
        MODEL_LABEL,
        "CLEAN RESULTS"
    )

    print(
        "========================================"
    )


    print(
        f"Accuracy : "
        f"{clean_accuracy * 100:.2f}"
    )


    print(
        f"Macro-F1 : "
        f"{clean_macro_f1 * 100:.2f}"
    )


    def prepare_perturbed_dataset(
        raw_dataframe
    ):

        model_df = (
            raw_dataframe.copy()
        )


        model_df[
            "text"
        ] = (

            model_df[
                "text"
            ]

            .apply(
                model_preprocess
            )
        )


        dataset = (

            Dataset

            .from_pandas(

                model_df,

                preserve_index=False
            )

            .map(

                tokenize_function,

                batched=True
            )
        )


        return dataset


    for dimension in DIMENSIONS:

        print(
            "\n\n"
            +
            "=" * 90
        )


        print(
            MODEL_LABEL,
            "-",
            dimension
        )


        print(
            "=" * 90
        )


        for severity in SEVERITIES:

            print(

                "\nSeverity:",
                int(
                    severity * 100
                ),
                "%"
            )


            for perturb_seed in (
                PERTURBATION_SEEDS
            ):


                key = (

                    dimension,
                    severity,
                    perturb_seed
                )


                raw_perturbed_df = (

                    PERTURBED_TEST_SETS[
                        key
                    ]
                )


                perturbed_dataset = (

                    prepare_perturbed_dataset(
                        raw_perturbed_df
                    )
                )


                perturbed_result = (

                    evaluate_dataset(

                        trainer,

                        perturbed_dataset
                    )
                )


                perturbed_accuracy = (

                    perturbed_result[
                        "accuracy"
                    ]
                )


                perturbed_macro_f1 = (

                    perturbed_result[
                        "macro_f1"
                    ]
                )


                delta_f1 = (

                    clean_macro_f1
                    -
                    perturbed_macro_f1
                )


                diagnostic = (

                    diagnostics_df[
                        (
                            diagnostics_df[
                                "Dimension"
                            ]
                            ==
                            dimension
                        )

                        &

                        (
                            diagnostics_df[
                                "Severity"
                            ]
                            ==
                            severity
                        )

                        &

                        (
                            diagnostics_df[
                                "Seed"
                            ]
                            ==
                            perturb_seed
                        )
                    ]

                    .iloc[0]
                )


                ALL_RESULTS.append({

                    "Model":
                        MODEL_LABEL,

                    "Dimension":
                        dimension,

                    "Severity":
                        severity,

                    "Severity_Percent":
                        int(
                            severity * 100
                        ),

                    "Perturbation_Seed":
                        perturb_seed,

                    "Clean_Accuracy":
                        clean_accuracy,

                    "Clean_Macro_F1":
                        clean_macro_f1,

                    "Perturbed_Accuracy":
                        perturbed_accuracy,

                    "Perturbed_Macro_F1":
                        perturbed_macro_f1,

                    "Delta_F1":
                        delta_f1,

                    "Realized_Rate":
                        diagnostic[
                            "Realized_Rate"
                        ],

                    "Changed_Instance_Rate":
                        diagnostic[
                            "Changed_Instance_Rate"
                        ]
                })


                print(

                    f"Seed {perturb_seed}"

                    f" | F1="
                    f"{perturbed_macro_f1 * 100:.2f}"

                    f" | Delta="
                    f"{delta_f1 * 100:.2f}"

                    f" | Realized="
                    f"{diagnostic['Realized_Rate'] * 100:.2f}%"

                    f" | Changed texts="
                    f"{diagnostic['Changed_Instance_Rate'] * 100:.2f}%"
                )


                del perturbed_dataset

                gc.collect()


    del trainer
    del model
    del tokenizer
    del train_dataset
    del clean_test_dataset


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    print(
        "\nFinished:",
        MODEL_LABEL
    )


results_df = pd.DataFrame(
    ALL_RESULTS
)


results_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "all_individual_runs.csv"
    ),

    index=False
)


summary_df = (

    results_df

    .groupby(
        [
            "Model",
            "Dimension",
            "Severity_Percent"
        ],

        as_index=False
    )

    .agg(

        Clean_Accuracy=(
            "Clean_Accuracy",
            "first"
        ),

        Clean_Macro_F1=(
            "Clean_Macro_F1",
            "first"
        ),

        Accuracy_Mean=(
            "Perturbed_Accuracy",
            "mean"
        ),

        Accuracy_SD=(
            "Perturbed_Accuracy",
            "std"
        ),

        Macro_F1_Mean=(
            "Perturbed_Macro_F1",
            "mean"
        ),

        Macro_F1_SD=(
            "Perturbed_Macro_F1",
            "std"
        ),

        Delta_F1_Mean=(
            "Delta_F1",
            "mean"
        ),

        Delta_F1_SD=(
            "Delta_F1",
            "std"
        ),

        Realized_Rate_Mean=(
            "Realized_Rate",
            "mean"
        ),

        Realized_Rate_SD=(
            "Realized_Rate",
            "std"
        ),

        Changed_Instance_Rate_Mean=(
            "Changed_Instance_Rate",
            "mean"
        ),

        N_Runs=(
            "Perturbation_Seed",
            "count"
        )
    )
)


summary_100 = (
    summary_df.copy()
)


scale_columns = [

    "Clean_Accuracy",
    "Clean_Macro_F1",

    "Accuracy_Mean",
    "Accuracy_SD",

    "Macro_F1_Mean",
    "Macro_F1_SD",

    "Delta_F1_Mean",
    "Delta_F1_SD",

    "Realized_Rate_Mean",
    "Realized_Rate_SD",

    "Changed_Instance_Rate_Mean"
]


for column in scale_columns:

    summary_100[
        column
    ] *= 100


clean_results_df = pd.DataFrame(
    CLEAN_RESULTS
)


clean_results_df[
    "Clean_Accuracy"
] *= 100


clean_results_df[
    "Clean_Macro_F1"
] *= 100


print(
    "\n\n"
    +
    "=" * 100
)

print(
    "CLEAN SARCASM DETECTION RESULTS"
)

print(
    "=" * 100
)


display(
    clean_results_df.round(3)
)


print(
    "\n\n"
    +
    "#" * 110
)

print(
    "FINAL RESULTS — ALL FIVE DATA-QUALITY DIMENSIONS"
)

print(
    "#" * 110
)


display(

    summary_100[
        [
            "Model",
            "Dimension",
            "Severity_Percent",

            "Clean_Macro_F1",

            "Macro_F1_Mean",
            "Macro_F1_SD",

            "Delta_F1_Mean",
            "Delta_F1_SD",

            "Accuracy_Mean",
            "Accuracy_SD",

            "Realized_Rate_Mean",

            "Changed_Instance_Rate_Mean",

            "N_Runs"
        ]
    ]

    .round(3)
)


paper_df = (
    summary_100.copy()
)


paper_df[
    "Macro-F1"
] = (

    paper_df[
        "Macro_F1_Mean"
    ]

    .map(
        lambda x:
        f"{x:.2f}"
    )

    +

    " ± "

    +

    paper_df[
        "Macro_F1_SD"
    ]

    .map(
        lambda x:
        f"{x:.2f}"
    )
)


paper_df[
    "Delta-F1"
] = (

    paper_df[
        "Delta_F1_Mean"
    ]

    .map(
        lambda x:
        f"{x:.2f}"
    )

    +

    " ± "

    +

    paper_df[
        "Delta_F1_SD"
    ]

    .map(
        lambda x:
        f"{x:.2f}"
    )
)


paper_df[
    "Accuracy"
] = (

    paper_df[
        "Accuracy_Mean"
    ]

    .map(
        lambda x:
        f"{x:.2f}"
    )

    +

    " ± "

    +

    paper_df[
        "Accuracy_SD"
    ]

    .map(
        lambda x:
        f"{x:.2f}"
    )
)


paper_df[
    "Realized Severity"
] = (

    paper_df[
        "Realized_Rate_Mean"
    ]

    .map(
        lambda x:
        f"{x:.2f}%"
    )
)


paper_table = (

    paper_df[
        [
            "Model",
            "Dimension",
            "Severity_Percent",
            "Macro-F1",
            "Accuracy",
            "Delta-F1",
            "Realized Severity"
        ]
    ]

    .sort_values(
        [
            "Model",
            "Dimension",
            "Severity_Percent"
        ]
    )
)


print(
    "\n\n"
    +
    "=" * 110
)

print(
    "PAPER-READY SARCASM TABLE"
)

print(
    "=" * 110
)


display(
    paper_table
)


dimension_ranking = (

    summary_100

    .groupby(
        [
            "Model",
            "Dimension"
        ],

        as_index=False
    )

    .agg(

        Mean_Delta_F1=(
            "Delta_F1_Mean",
            "mean"
        )
    )

    .sort_values(
        [
            "Model",
            "Mean_Delta_F1"
        ],

        ascending=[
            True,
            False
        ]
    )
)


print(
    "\n\n"
    +
    "=" * 100
)

print(
    "SARCASM — QUALITY-DIMENSION SENSITIVITY RANKING"
)

print(
    "=" * 100
)


display(
    dimension_ranking.round(3)
)


cross_model_summary = (

    summary_100

    .groupby(
        [
            "Dimension",
            "Severity_Percent"
        ],

        as_index=False
    )

    .agg(

        Mean_Delta_F1=(
            "Delta_F1_Mean",
            "mean"
        ),

        Mean_Macro_F1=(
            "Macro_F1_Mean",
            "mean"
        )
    )
)


print(
    "\n\n"
    +
    "=" * 100
)

print(
    "CROSS-MODEL SARCASM SUMMARY"
)

print(
    "=" * 100
)


display(
    cross_model_summary.round(3)
)


print(
    "\n\n"
    +
    "=" * 100
)

print(
    "SEVERITY MONOTONICITY CHECK"
)

print(
    "=" * 100
)


for model_label in MODELS.keys():

    for dimension in DIMENSIONS:

        subset = (

            summary_100[
                (
                    summary_100[
                        "Model"
                    ]
                    ==
                    model_label
                )

                &

                (
                    summary_100[
                        "Dimension"
                    ]
                    ==
                    dimension
                )
            ]

            .sort_values(
                "Severity_Percent"
            )
        )


        deltas = (

            subset[
                "Delta_F1_Mean"
            ]

            .values
        )


        monotonic = all(

            deltas[i]
            <=
            deltas[i + 1]

            for i in range(
                len(deltas) - 1
            )
        )


        print(

            f"{model_label:25s}"

            f" | {dimension:20s}"

            f" | Monotonic = {monotonic}"
        )


severity_30_ranking = (

    summary_100[
        summary_100[
            "Severity_Percent"
        ]
        ==
        30
    ]

    [
        [
            "Model",
            "Dimension",
            "Delta_F1_Mean"
        ]
    ]

    .sort_values(
        [
            "Model",
            "Delta_F1_Mean"
        ],

        ascending=[
            True,
            False
        ]
    )
)


print(
    "\n\n"
    +
    "=" * 100
)

print(
    "SENSITIVITY AT 30% SEVERITY"
)

print(
    "=" * 100
)


display(
    severity_30_ranking.round(3)
)


summary_100.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "five_dimensions_summary.csv"
    ),

    index=False
)


paper_table.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "paper_ready_sarcasm_results.csv"
    ),

    index=False
)


dimension_ranking.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "dimension_sensitivity_ranking.csv"
    ),

    index=False
)


cross_model_summary.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "cross_model_summary.csv"
    ),

    index=False
)


severity_30_ranking.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "severity_30_ranking.csv"
    ),

    index=False
)


clean_results_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "clean_model_results.csv"
    ),

    index=False
)


diagnostics_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "all_perturbation_diagnostics.csv"
    ),

    index=False
)


print(
    "\n\n"
    +
    "#" * 100
)

print(
    "SARCASM EXPERIMENT COMPLETED"
)

print(
    "#" * 100
)


print(
    "\nTask: Sarcasm Detection"
)

print(
    "Input column: text"
)

print(
    "Target column: sarcastic"
)


print(
    "\nModels:"
)

print(
    "1. AraBERTv2"
)

print(
    "2. Twitter-RoBERTa-Irony"
)


print(
    "\nDimensions:"
)

for dimension in DIMENSIONS:

    print(
        "-",
        dimension
    )


print(
    "\nMain results file:"
)

print(

    os.path.join(
        OUTPUT_DIR,
        "paper_ready_sarcasm_results.csv"
    )
)


print(
    "\nFINAL PAPER-READY RESULTS:"
)


display(
    paper_table
)
